# Locate the *hardest* attack and let Stable Diffusion repair it

The tile gate in [`../SD-Noise-Gate/`](../SD-Noise-Gate/) flags a tile by its **high-frequency
energy** `mean(|tile - blur(tile)|)`. Against the attack zoo in
[`../Attack-Benchmark/`](../Attack-Benchmark/) that score is well matched to the **L-inf**
attacks (FGSM / PGD / Square) that inject loud high-frequency noise. The one attack the zoo
marks *"evades HF detector"* is the **low-frequency / Nightshade-Glaze** poison: its energy
sits in *low* spatial frequencies, so `|tile - blur(tile)|` barely moves and the gate is blind
to it. That is the **hardest attack for this detector**, so it is the one we tackle here.

**Best method to still find *where* it hit.** Replace the single high-frequency band with a
small **bank of octave band-pass filters** (a multi-band spectral residual). Each tile is
robustly z-scored *per band* against the trusted-clean calibration set, and we take the **max
across bands**. The old HF score is just the top band of this bank, so it is a strict
generalization: L-inf attacks still light up the top band, and the low-frequency poison now
lights up a *lower* band. The flagged tiles give us a mask.

**Repair.** Feed that mask to **Stable Diffusion inpainting**, which regenerates only the
attacked region and leaves the rest of the picture untouched.

```
image --> [ multi-band tile scan ] --clean--> keep
                 |
                 +--attacked tiles--> mask --> SD inpaint --> repaired image
```

Environment: Kaggle **GPU T4 x2, Internet On** (falls back to web sample images if no dataset
is mounted; the SD repair step is guarded so the notebook still runs without the weights).

## 0. Install & imports

In [ ]:
!pip install -q diffusers transformers accelerate

In [ ]:
import torch, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image, ImageDraw, ImageFilter
import urllib.request, os, glob

N_GPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(N_GPU)] if N_GPU else ["cpu"]
_D = DEVICES[0]
print("Using devices:", DEVICES)

## 1. Tiler (reused from `SD-Noise-Gate`)

Images are `[1,3,512,512]` float tensors in **[0,1] pixel space**. `tile_image` cuts one image
into a row-major `[16,3,128,128]` stack, exactly as the gate does.

In [ ]:
SIZE, GRID = 512, 4            # SD-native resolution, 4x4 = 16 tiles/image
TILE = SIZE // GRID            # 128 px per tile
to_tensor = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

def load_image(path_or_url):
    if str(path_or_url).startswith("http"):
        fn = "/tmp/" + os.path.basename(path_or_url)
        if not os.path.exists(fn): urllib.request.urlretrieve(path_or_url, fn)
        path_or_url = fn
    return to_tensor(Image.open(path_or_url).convert("RGB")).unsqueeze(0)

def tile_image(x, grid=GRID, tile=TILE):
    # [1,3,SIZE,SIZE] -> [grid*grid, 3, tile, tile]  (row-major tile order)
    p = x.unfold(2, tile, tile).unfold(3, tile, tile)
    return p.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, tile, tile).contiguous()

def tiles_to_full(per_tile, grid=GRID, tile=TILE):
    # [16] per-tile values -> [SIZE,SIZE] blocky map for overlay
    return np.kron(np.asarray(per_tile).reshape(grid, grid), np.ones((tile, tile)))

def to_np(t): return t.squeeze().detach().cpu().permute(1, 2, 0).numpy()

## 2. Two detectors: the old HF band vs. the multi-band bank

A Gaussian blur at scale `sigma` keeps frequencies *below* `1/sigma`. Subtracting two blurs at
neighbouring scales gives a **band-pass** slice; a stack of them from fine to coarse covers the
spectrum in octaves.

- **`hf_energy`** = the gate's original score = the single finest band `|tile - blur_1|`.
- **`band_energies`** = that finest band **plus** three successively coarser bands.

We calibrate a robust `median + k*MAD` baseline **per band** on trusted-clean tiles, z-score every
incoming tile against it, and flag a tile if **any** band exceeds `k`. The HF detector is the
same rule restricted to band 0, which is why it misses a purely low-frequency poison.

In [ ]:
def _gauss(sigma, ksize=None):
    ksize = ksize or (int(6*sigma) | 1)          # odd kernel wide enough for sigma
    ax = torch.arange(ksize) - ksize // 2
    g = torch.exp(-(ax**2) / (2*sigma**2)); g = g / g.sum()
    return torch.outer(g, g).view(1, 1, ksize, ksize).repeat(3, 1, 1, 1)  # depthwise, 3ch

_SIGMAS = [1.0, 2.0, 4.0, 8.0]                    # fine -> coarse; band i = blur_i - blur_{i+1}
_BANK = [_gauss(s) for s in _SIGMAS]
N_BANDS = len(_BANK)                              # 4 bands: 1 HF (top) + 3 lower

def _blur(t, k): return F.conv2d(t, k.to(t.device, t.dtype), padding=k.shape[-1]//2, groups=3)

def hf_energy(tiles):
    # OLD gate score: energy of the single finest band only -> [B] numpy
    return (tiles - _blur(tiles, _BANK[0])).abs().mean(dim=(1, 2, 3)).float().cpu().numpy()

def band_energies(tiles):
    # NEW: energy in each octave band -> [B, N_BANDS] numpy
    bl = [_blur(tiles, k) for k in _BANK]
    bands = [tiles - bl[0]] + [bl[i] - bl[i+1] for i in range(len(bl)-1)]  # HF ... LF
    return torch.stack([b.abs().mean(dim=(1, 2, 3)) for b in bands], 1).float().cpu().numpy()

def calibrate(clean_images, k=3.0):
    # per-band robust baseline over trusted-clean tiles; returns (med, scale, k)
    E = np.concatenate([band_energies(tile_image(x).to(_D)) for x in clean_images])  # [Nc*16, B]
    med = np.median(E, 0); mad = np.median(np.abs(E - med), 0) + 1e-8
    return med, 1.4826 * mad, k

def scan(x, calib):
    # per-tile z-score in every band -> (z_multiband[16], z_hf[16])
    med, scale, k = calib
    z = (band_energies(tile_image(x).to(_D)) - med) / scale   # [16, B]
    return z.max(1), z[:, 0], k                               # max-over-bands, HF-only, threshold

print(f"detectors ready: HF = band 0 of a {N_BANDS}-band bank (sigmas {_SIGMAS})")

## 3. The hardest attack: a low-frequency (Nightshade-Glaze-flavoured) poison

Standard PGD adds the **sign** of the gradient, which is broadband and noisy (loud in the HF
band). Here we instead **low-pass the perturbation** at every step, so the injected energy is
forced into *low* frequencies. It stays a valid `[0,1]` image inside an L-inf `epsilon` budget
and is confined to a random irregular blob (not tile-aligned — finding *which* tiles it touched
is the detector's job). A ResNet50 only supplies the gradient; the detectors never see it.

In [ ]:
_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def _normalize(t): return (t - _mean.to(t.device)) / _std.to(t.device)
_ATK = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).eval().to(_D)

def random_blob_mask(size=SIZE, seed=None, n_verts=9, rad_frac=(0.12, 0.34)):
    # irregular polygon -> [1,1,size,size] in {0,1}
    rng = np.random.default_rng(seed)
    cx, cy = rng.uniform(0.30, 0.70, 2) * size
    angles = np.sort(rng.uniform(0, 2*np.pi, n_verts))
    radii = rng.uniform(*rad_frac, n_verts) * size
    pts = [(float(cx + r*np.cos(a)), float(cy + r*np.sin(a))) for a, r in zip(angles, radii)]
    im = Image.new("L", (size, size), 0); ImageDraw.Draw(im).polygon(pts, fill=1)
    return torch.tensor(np.array(im), dtype=torch.float32).view(1, 1, size, size)

def _raw_grad(xi):
    out = _ATK(_normalize(xi))
    loss = F.cross_entropy(out, out.argmax(1).detach())   # push away from current prediction
    _ATK.zero_grad(); loss.backward()
    return xi.grad.detach()

def lowfreq_poison(x, epsilon=0.06, steps=12, alpha=0.02, sigma=8.0, mask=None):
    # PGD whose per-step update is LOW-PASS filtered -> energy concentrated in low frequencies
    x0 = x.clone().detach().to(_D); xi = x0.clone()
    k = _gauss(sigma).to(_D)
    m = mask.to(_D) if mask is not None else None
    for _ in range(steps):
        xi.requires_grad_(True)
        g = _raw_grad(xi)
        with torch.no_grad():
            g = _blur(g, k)                       # <-- the low-frequency twist
            g = g / (g.abs().amax() + 1e-8)       # rescale so alpha is a real step size
            step = alpha * g
            if m is not None: step = step * m
            xi = torch.min(torch.max(xi + step, x0 - epsilon), x0 + epsilon)
            xi = torch.clamp(xi, 0, 1)
    return xi.detach().cpu()

def true_tiles(mask, cover=0.05):
    # ground truth: tile is 'attacked' if the blob covers > cover of it
    mt = tile_image(mask.repeat(1, 3, 1, 1))
    return mt.mean(dim=(1, 2, 3)).numpy() > cover

## 4. Get images, calibrate, and poison one

Point `IMAGE_SOURCE` at your own Kaggle dataset (`"folder"`) or use the built-in web samples.
The first 4 images are the **trusted-clean calibration set**; the next one is poisoned with the
low-frequency attack so we can measure detection against known ground truth.

In [ ]:
IMAGE_SOURCE = "web"                 # "web" (samples) or "folder" (your Kaggle dataset)
IMAGE_DIR    = "/kaggle/input"       # used when IMAGE_SOURCE == "folder"
EPSILON, K_SENSITIVITY = 0.06, 3.0   # L-inf poison budget; detector threshold (lower = stricter)

_WEB = [
    "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
    "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n07747607_orange.JPEG",
    "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n02690373_airliner.JPEG",
    "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n03095699_container_ship.JPEG",
    "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n04285008_sports_car.JPEG",
    "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n07753592_banana.JPEG",
]

if IMAGE_SOURCE == "folder":
    paths = sorted(f for f in glob.glob(os.path.join(IMAGE_DIR, "**", "*"), recursive=True)
                   if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))[:6]
    raw = [load_image(p) for p in paths]
else:
    raw = [load_image(u) for u in _WEB]
assert len(raw) >= 5, "need >= 5 images (4 calibrate + 1 test)"

calib = calibrate(raw[:4], k=K_SENSITIVITY)                 # calibrate on trusted-clean
mask  = random_blob_mask(seed=0)
clean = raw[4]
attacked = lowfreq_poison(clean, epsilon=EPSILON, mask=mask)
gt = true_tiles(mask)
print(f"calibrated on 4 clean images; poisoned 1 image, {gt.sum()} tiles truly attacked")

## 5. Which detector finds the attacked area?

We score the poisoned image with both detectors and compare the flagged tiles to ground truth
using **IoU** (overlap of flagged vs. truly-attacked tiles). The low-frequency poison should
leave the HF detector near-blind while the multi-band bank localizes it.

In [ ]:
def iou(pred, true):
    inter = (pred & true).sum(); union = (pred | true).sum()
    return inter / union if union else 1.0

z_mb, z_hf, k = scan(attacked, calib)
flag_mb, flag_hf = z_mb > k, z_hf > k
print(f"HF detector      : flagged {flag_hf.sum():2d} tiles | IoU vs truth = {iou(flag_hf, gt):.2f}")
print(f"Multi-band bank  : flagged {flag_mb.sum():2d} tiles | IoU vs truth = {iou(flag_mb, gt):.2f}")

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(to_np(clean));    ax[0].set_title("clean")
ax[1].imshow(to_np(attacked)); ax[1].set_title("low-freq poisoned")
ax[2].imshow(to_np(attacked)); ax[2].imshow(tiles_to_full(z_hf), alpha=0.5, cmap="hot")
ax[2].set_title(f"HF score  (IoU {iou(flag_hf, gt):.2f})")
ax[3].imshow(to_np(attacked)); ax[3].imshow(tiles_to_full(z_mb), alpha=0.5, cmap="hot")
ax[3].set_title(f"multi-band  (IoU {iou(flag_mb, gt):.2f})")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 6. Repair the located region with Stable Diffusion inpainting

Turn the multi-band flags into a pixel mask (dilated a little so the inpainter blends at the
edges) and let `StableDiffusionInpaintPipeline` regenerate **only** those tiles. Everything
outside the mask is copied through untouched, so a clean image would pass through unchanged.
The step is guarded: without the diffusers weights it just reports the mask it *would* repair.

In [ ]:
# multi-band flags -> 512x512 {0,255} mask, dilated by ~half a tile for clean seams
mask_np = (tiles_to_full(flag_mb.astype(np.float32)) * 255).astype(np.uint8)
mask_img = Image.fromarray(mask_np).filter(ImageFilter.MaxFilter(TILE // 2 * 2 + 1))
attacked_img = Image.fromarray((to_np(attacked) * 255).astype(np.uint8))

repaired = None
if flag_mb.any():
    try:
        from diffusers import StableDiffusionInpaintPipeline
        inpaint = StableDiffusionInpaintPipeline.from_pretrained(
            "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16).to(_D)
        inpaint.set_progress_bar_config(disable=True)
        repaired = inpaint(prompt="a natural, photorealistic photo, high quality",
                           image=attacked_img, mask_image=mask_img,
                           num_inference_steps=30, guidance_scale=7.5).images[0]
        print("repaired the flagged region with SD inpainting")
    except Exception as e:
        print("SD inpaint unavailable, showing the mask only:", type(e).__name__)
else:
    print("no tiles flagged — nothing to repair")

n = 3 if repaired is None else 4
fig, ax = plt.subplots(1, n, figsize=(4*n, 4))
ax[0].imshow(attacked_img); ax[0].set_title("attacked")
ax[1].imshow(mask_img, cmap="gray"); ax[1].set_title("located repair mask")
ax[2].imshow(attacked_img); ax[2].imshow(mask_np, alpha=0.4, cmap="Reds"); ax[2].set_title("mask over image")
if repaired is not None: ax[3].imshow(repaired); ax[3].set_title("SD-repaired")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Takeaway

The low-frequency poison is the attack the HF gate was blind to (low IoU); widening the single
HF band into a **multi-band spectral residual** recovers *where* it hit (high IoU) at nearly no
extra cost — it is still forward-only and calibration-only, with no new training. Those flagged
tiles then drive **Stable Diffusion inpainting** to rebuild just the attacked region.

Honest limits: natural images carry a lot of genuine low-frequency variation, so the coarse
bands are noisier than the HF band — expect more false positives there, tunable via
`K_SENSITIVITY` and the `_SIGMAS` bank. A learned per-tile classifier is the next step
([`../SD-Noise-Gate/`](../SD-Noise-Gate/) research log).